# 008 — Qualification du dataset jardins
Profil configurable. Les modifications sont des hypothèses synthétiques à revoir, jamais des observations. Aucun appel API.

In [ ]:
from pathlib import Path
import sys, json, copy, hashlib
import pandas as pd
cwd = Path.cwd().resolve()
ROOT = next(p for p in (cwd, *cwd.parents) if (p / "gdai-plant/config/dataset-scope-idf-v1.json").is_file())
PLANT = ROOT / "gdai-plant"
sys.path.insert(0, str(PLANT))
from matching.dataset_quality import audit
PROFILE = PLANT / "config/dataset-scope-idf-v1.json"
SOURCE = PLANT / "data/scenarios/garden-scenarios-climate-v1.json"
PLAN = SOURCE.with_name("garden-idf-review-plan-v1.json")
OUTPUT = SOURCE.with_name("garden-scenarios-idf-v1.json")
REPORT = SOURCE.with_name("garden-idf-qualification-report-v1.json")
profile = json.loads(PROFILE.read_text())
raw = SOURCE.read_bytes()
document = json.loads(raw)
report = audit(document, profile)
display(pd.DataFrame(report["issues"]))
print(report["flagged"], "/", report["total"])

## Revue explicite
Le plan contient tous les scénarios, y compris ceux sans anomalie automatique. Modifier les champs dans `changes`, documenter la raison et passer `reviewed` à true après revue géographique et horticole. Ne pas modifier les identifiants ni la provenance originale.

In [ ]:
if not PLAN.exists():
    plan = [{"id": row["id"], "changes": {}, "reason": "", "reviewed": False}
            for row in document["scenarios"]]
    with PLAN.open("x", encoding="utf-8") as f:
        json.dump(plan, f, ensure_ascii=False, indent=2)
plan = json.loads(PLAN.read_text())
display(pd.DataFrame(plan))
print("Reviewed:", sum(p["reviewed"] is True for p in plan))

In [ ]:
plan = json.loads(PLAN.read_text())
ids = [r["id"] for r in document["scenarios"]]
assert len(plan) == len(ids) == len(set(ids))
assert {p["id"] for p in plan} == set(ids)
assert all(p["reviewed"] is True and p["reason"].strip() for p in plan)
updated = copy.deepcopy(document)
lookup = {p["id"]: p for p in plan}
allowed = set(profile["allowed_values"]) | set(profile["numeric_ranges"]) | {"region", "plausibility_reason"}
for row in updated["scenarios"]:
    p = lookup[row["id"]]
    assert set(p["changes"]) <= allowed
    row.update(p["changes"])
final = audit(updated, profile)
assert final["flagged"] == 0 and final["target_count_met"], final
assert SOURCE.read_bytes() == raw
updated["qualification"] = {"profile_id": profile["profile_id"], "status": "SYNTHETIC_REVIEWED", "source_sha256": hashlib.sha256(raw).hexdigest(), "profile_sha256": hashlib.sha256(PROFILE.read_bytes()).hexdigest()}
assert not OUTPUT.exists() and not REPORT.exists()
encoded = json.dumps(updated, ensure_ascii=False, indent=2)
final["output_sha256"] = hashlib.sha256(encoded.encode()).hexdigest()
final["review_plan"] = plan
OUTPUT.write_text(encoded, encoding="utf-8")
REPORT.write_text(json.dumps(final, ensure_ascii=False, indent=2), encoding="utf-8")
print(OUTPUT, REPORT)